# Memory-Based Collaborative Filtering: Item-Item CF

**Memory-based** = dùng *trực tiếp* ma trận user-item mỗi lần gợi ý, **không có bước train**.

Có 2 biến thể, khác nhau ở chiều so sánh trên ma trận:

| | So sánh cái gì | Câu hỏi |
|---|---|---|
| **User-User** (xem `cf_demo.ipynb`) | các cột (user) | "Ai có gu giống User1? Họ chấm Item3 bao nhiêu?" |
| **Item-Item** (notebook này) | các hàng (item) | "Item3 giống item nào User1 đã chấm? User1 chấm những item đó bao nhiêu?" |

Item-item (cách Amazon dùng) thường được ưa chuộng hơn user-user vì số item ít hơn số user và "gu" của item ổn định hơn gu của người.

In [1]:
import pandas as pd
import numpy as np

# Ma tran User-Item: hang = item, cot = user, NaN = chua danh gia
data = {
    'User1': [5, 4, np.nan, 2, 1],
    'User2': [4, 5, 3, np.nan, 2],
    'User3': [np.nan, 2, 5, 4, 3],
    'User4': [3, 4, 2, 5, np.nan],
}
df = pd.DataFrame(data, index=['Item1', 'Item2', 'Item3', 'Item4', 'Item5'])
df

,User1,User2,User3,User4
Item1,5.0,4.0,NaN,3.0
Item2,4.0,5.0,2.0,4.0
Item3,NaN,3.0,5.0,2.0
Item4,2.0,NaN,4.0,5.0
Item5,1.0,2.0,3.0,NaN


## Bước 1: Tính độ tương đồng giữa các ITEM

Dùng **cosine similarity** giữa 2 hàng (2 item), nhưng chỉ tính trên các user mà **cả hai item đều được chấm điểm** — đây là cách xử lý NaN đúng hơn so với `fillna(0)` (điền 0 sẽ làm hệ thống hiểu nhầm "chưa xem" thành "rất ghét").

$$\text{sim}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

In [2]:
def cosine_sim(a, b):
    """Cosine similarity tren cac vi tri ca hai vector deu co gia tri."""
    mask = a.notna() & b.notna()
    if mask.sum() < 2:
        return 0.0
    a, b = a[mask], b[mask]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

items = df.index
item_sim = pd.DataFrame(
    [[cosine_sim(df.loc[i], df.loc[j]) for j in items] for i in items],
    index=items, columns=items,
)
item_sim.round(3)

,Item1,Item2,Item3,Item4,Item5
Item1,1.000,0.974,0.998,0.796,0.908
Item2,0.974,1.000,0.798,0.894,0.797
Item3,0.998,0.798,1.000,0.870,0.999
Item4,0.796,0.894,0.870,1.000,0.990
Item5,0.908,0.797,0.999,0.990,1.000


## Bước 2: Dự đoán điểm cho item chưa được đánh giá

Để dự đoán User1 chấm Item3:
1. Lấy các item User1 **đã** chấm.
2. Trong số đó, chọn top-k item **giống Item3 nhất**.
3. Dự đoán = trung bình có trọng số điểm User1 đã chấm, trọng số là độ tương đồng:

$$\hat{r}_{u,i} = \frac{\sum_{j \in \text{top-}k} \text{sim}(i,j) \cdot r_{u,j}}{\sum_{j \in \text{top-}k} \text{sim}(i,j)}$$

In [3]:
def predict_rating(user, item, k=2):
    """Du doan diem cua user cho item bang item-item CF."""
    rated_items = df[user].dropna().index          # item user da cham
    if len(rated_items) == 0:
        return np.nan
    sims = item_sim.loc[item, rated_items]          # do giong cua item voi tung item da cham
    top_k = sims.nlargest(k)
    if top_k.sum() == 0:
        return np.nan
    ratings = df.loc[top_k.index, user]
    return np.dot(top_k, ratings) / top_k.sum()

pred = predict_rating('User1', 'Item3')
print(f"Du doan User1 cham Item3: {pred:.3f}")

Du doan User1 cham Item3: 3.000


## Bước 3: Gợi ý top-n item

Lặp bước 2 cho **mọi item user chưa chấm**, sắp xếp theo điểm dự đoán giảm dần.

In [4]:
def recommend_items(user, n_recommendations=3, k=2):
    user_ratings = df[user]
    items_to_predict = user_ratings[user_ratings.isna()].index

    predicted = []
    for it in items_to_predict:
        p = predict_rating(user, it, k)
        if not np.isnan(p):
            predicted.append((it, p))

    predicted.sort(key=lambda x: x[1], reverse=True)
    return predicted[:n_recommendations]

for user in df.columns:
    recs = recommend_items(user)
    print(f"Goi y cho {user}: " + ", ".join(f"{it} ({p:.2f})" for it, p in recs))

Goi y cho User1: Item3 (3.00)
Goi y cho User2: Item4 (3.42)
Goi y cho User3: Item1 (3.52)
Goi y cho User4: Item5 (3.49)


## Nhận xét

**Ưu điểm:**
- Đơn giản, không cần train, dễ giải thích ("vì bạn thích Item1 và Item5 nên gợi ý Item3").
- Gợi ý cập nhật ngay khi có rating mới (chỉ cần tính lại similarity).

**Nhược điểm:**
- **Khả năng mở rộng:** mỗi lần dự đoán phải quét lại ma trận gốc — với hàng triệu user/item, chi phí tính similarity là $O(n^2)$.
- **Data sparsity:** nếu 2 item ít được chấm chung bởi cùng user, similarity không đáng tin.
- Cosine trên rating thô (toàn số dương 1–5) làm mọi item trông giống nhau (similarity toàn ~0.8–1.0). Thực tế dùng **adjusted cosine**: trừ điểm trung bình của từng *user* trước khi tính.
